# Machine learning: teaching a computer to recognise a genre

## What is different here

Everything we have done so far, we told the computer **how** to perform a task. We wrote the rules it should follow to accomplish a specfic goal: *count this word*, *divide by that length*, *sort*.

We instruct the computer what to do, which rules to follow. We write the algorithm for them.

This works, and can be very effective, for simple and straightforward problems.

However, the real world is complex, and detailing which rules a computer should follow becomes rapidly unfeasible.

Could you program a self-driving car with `if ... else` rules? Or even a genre detection tool?

In machine learning we flip things around. We give the computer **examples with the right answer attached**, and ask it to figure out the rule itself.

We use an algorithm to establish the rules instead of compiling these rules ourselves.


### Examples of Machine Learning Applications in Digital Humanities
- **Document classification**: Sentiment, Ideology, Metadata extraction -> we will focus on this!
- **Word classification**: Named Entity Recognition, Part-of-Speech tagging, ...

![ner](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/ner_concrete_example.png)

## Classification: What do you need


In this notebook, we will want to build an automatic classifier. To start, you'll need the following components:

* **texts** — what the model gets to look at (here: the words in the title)
* **label** — what it has to predict (here: the genre)

Supervised learning is nothing more than: *learn the relationship between words in the texts and labels from examples, then apply it to examples you have never seen*.

![mlwf](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/supervised_workflow.png)

In [ ]:
!wget -q https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/main/Sessions/data/bl_books_genre.csv

In [ ]:
import pandas as pd

df = pd.read_csv("bl_books_genre.csv")
print(df.shape)
df.head(3)

## Preparing the data

Two decisions before we start, both of which matter more than any model choice.

**First**, we keep only the books labelled *Fiction* or *Non-fiction*, dropping the handful marked "Both" or "Can't tell".



**Second**, we keep only the **English** books. Remember what we found in the last notebook: the Danish and Swedish books are almost all non-fiction. If we left them in, a model could score well by learning "Scandinavian word ⇒ non-fiction" — which is not genre recognition at all. We come back to this at the end.

**Question**: What is the importance of these choices, how will it affect the classifier's capability? How would you document this?

In [ ]:
import numpy as np
df['length'] = df.title.apply(lambda x: len(x.split()))
df['av_word_length'] = df.title.apply(lambda x: np.mean([len(w) for w in x.split()]))

In [ ]:
books = df[df["genre"].isin(["Fiction", "Non-fiction"])]
books = books[books["language"] == "English"]

print("books:", len(books))
print(books["genre"].value_counts())

In [ ]:
sns.scatterplot(x="length", y = "av_word_length", data=books,hue='genre', alpha=.5)

## Splitting into training and test data

**Why split the data?** Why don't we use all examples for training, the more data the better right?

Well, not really. Besides creating a genre classifier, we need to establish how well it works: **we need to evaluate it!**

If we test the model on the same titles it learned from, we can't tell whether it has actually learned a **general pattern** or has simply **memorized** the answers. To check this honestly, we hold some examples back.

**Question**: why is it important to learn a general pattern? Why is memorization a problem for machine learning?

![](https://scikit-learn.org/stable/_images/sphx_glr_plot_underfitting_overfitting_001.png)

For more on overfitting, see [here](https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html)

### How much should we set aside?

**Training set** — the examples the model learns from (typically 70–80% of the data).

**Test set** — the examples set aside and never shown to the model during training, used afterward to check how well it predicts labels it hasn't seen.

Only performance on the **test set** tells us how the model would do on genuinely new titles — which is the whole point.


This is the most important rule in machine learning:

> **Never evaluate a model on the examples it learned from.**

A model that has seen an example can simply memorise its answer. To find out whether it has learned anything general, we hide some data from it and test on that.

### In Python

`train_test_split` does this for us. `stratify=y` keeps the fiction/non-fiction proportion the same in both splits, and `random_state=42` makes the split reproducible — you and your neighbour will get identical results.

In [ ]:
from sklearn.model_selection import train_test_split

X = books["title"]                            # the features: the raw titles
y = (books["genre"] == "Fiction").astype(int) # the label: 1 for fiction, 0 for non-fiction

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("training examples:", len(X_train))
print("test examples:    ", len(X_test))
print('fiction in train set:'f"{y_train.mean():.0%}")
print("fiction in test set:", f"{y_test.mean():.0%}")

## From titles to numbers

A model cannot read text, it needs numbers — and we already know how to turn a text into numbers, because that is exactly what we did yesterday: **count the words**.

`CountVectorizer` does that in one line: it builds a vocabulary from all the titles, then represents every title as a row of word counts. Let's see it on five titles before turning it loose on the whole training set.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

sample_titles = X_train.iloc[:5].tolist()

count_vectorizer = CountVectorizer() # instantiate a vectorizer
sample_counts = count_vectorizer.fit_transform(sample_titles) # convert titles to the document-tern matrix

dtm = pd.DataFrame(
    sample_counts.toarray(),
    columns=count_vectorizer.get_feature_names_out(),
    index=sample_titles,
)
dtm

In [ ]:
import seaborn as sns
sns.heatmap(dtm)

Every row is a title, every column a word that appears in at least one of these five titles, and every cell is how many times that word occurs in that title. This is the **document-term matrix** — the same bag-of-words idea as yesterday's `Counter`, just built for every document and every word in the vocabulary at once, instead of one document and one word at a time.

![bow](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/bag_of_words.png)


### Features and Representations

As mentioned, the model doesn't read text as a string of characters, to use it for classification we have to convert it to a numerical representation or a vector.

Put simply: A computer can't read the way we do — it can't sense that "Mysteries" evokes suspense the way a human reader can. Instead, we have to translate the text into something a computer can work with: numbers.

This translation step is called document **representation**, and the numbers it produces are the **features**.

### Bag-of-Words representation

The simplest way to represent a title is to ignore word order and grammar entirely, and just ask: which words are present? This is called a "bag of words" — imagine tipping all the words from a title into a bag and shaking it up, so all that's left is which words are in there and how many times.

Take our two examples from before:

"The Mysteries of Udolpho" → contains the words: the, mysteries, of, udolpho
"A Voyage Round the World" → contains the words: a, voyage, round, the, world

To turn these into numbers, we build a big list of every word that appears anywhere across the 630 training titles — this list of unique words is called the vocabulary. 

Each title then becomes a row of numbers (or vector), one number per word in the vocabulary, recording how many times that word appears in this title (usually 0, since most titles use only a handful of words from a vocabulary that might contain several hundred).

So "mysteries" might be feature #412 in our list, and it's a 1 for the Udolpho title and a 0 for the Voyage title. "Voyage" might be feature #603, and it's the reverse.


### The bigger picture: representation is a choice

Bag of words is only one way to represent a document. 

### Question: why might bag-of-words not be a good idea?

It's simple and often surprisingly effective for genre detection, but it throws away a lot: word order, sentence structure, meaning, context. 

Below we look at another method for counting: Tf-Idf.


### Counting and Weighing

Raw counts have the same problem we met in notebook 2g, though: a common word like `the` counts for as much as a rare, informative one, and a long title racks up bigger counts than a short one for no meaningful reason.

`TfidfVectorizer` fixes exactly that. It does the whole of notebook 2g in one line: builds the vocabulary, counts the words in every document, and **weights** each word by how rare it is across the collection (inverse document frequency).

Rare words will have higher values, the importance of very frequents will be dampened.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", 
                             max_df=0.5,
                             min_df=2,
                             token_pattern=r"(?u)\b\w\w+\b",
                             ngram_range=(1, 2))


X_train_vectors = vectorizer.fit_transform(X_train)
X_test_vectors = vectorizer.transform(X_test)

print("shape of the training matrix:", X_train_vectors.shape)
print("shape of the testing matrix:", X_test_vectors.shape)

### Tuning the vectorizer

Four extra arguments make the vocabulary smaller and more informative:

* **`stop_words="english"`** — drops common English function words (*the*, *of*, *a*, *and*...) before building the vocabulary. They occur in almost every title and carry no information about genre.
* **`max_df=0.5`** — also ignore any word that appears in more than 50% of the titles. Even words `stop_words` doesn't know about can be so common in this particular corpus that they are just as uninformative.
* **`min_df=2`** — ignore any word that appears in fewer than 2 titles. A word used only once is almost certainly a typo, a name, or too rare to help the model generalise to titles it has not seen.
* **`ngram_range=(1, 2)`** — keep not just single words (*unigrams*) but also pairs of consecutive words (*bigrams*). Bag-of-words normally throws away word order entirely; this lets a phrase like "voyage round" survive as its own feature, distinct from "voyage" and "round" on their own.

Together, these trade a little recall for a much smaller, cleaner vocabulary — worth trying whenever the plain vectorizer's feature count feels too large or too noisy.

In [ ]:
vectorizer.get_feature_names_out()[100:150]

Read that shape carefully: **630 rows** — one per book — and **704 columns, one per word (or word pair) left in the vocabulary after filtering**.

Every book is now a row of numbers, almost all of them zero, with values only in the columns for the words its title happens to use. This is the **document-term matrix**: the bag-of-words idea from day 2, written out as a table.

⚠️ Note the two different method names, because this trips everybody up:
* `.fit_transform()` on the **training** data: work out the vocabulary *and* convert.
* `.transform()` on the **test** data: convert using the vocabulary already learnt.

If we called `.fit_transform()` on the test set, it would build its vocabulary from data the model is not allowed to see. The test set must stay sealed.

## Model 1: k-nearest neighbours

![knn](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/knn_genre_classification.png)

**k-nearest neighbours**  classifies a new example by finding the labeled examples most similar to it, and let them vote.

Similarity or proximity is measured using the features — count or tf-idf vectors we created earlier. 
Titles that share more words will sit closer together, titles that share fewer words sit farther apart. Cosine similarity is a common way to measure this closeness between two vectors, by looking at the respective angle between to examples.

To classify a book we have never seen: find the `k` most similar titles among the ones we do have labels for, and give it whatever label the majority of them have.

kNN  doesn't involve training in any real sense — the model just keeps the examples, and does all its work at prediction time.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
knn.fit(X_train_vectors, y_train)

predictions = knn.predict(X_test_vectors)
print(predictions[:20])

`1` means fiction, `0` non-fiction. How many did it get right?

In [ ]:
from sklearn.metrics import accuracy_score

print("accuracy:", round(accuracy_score(y_test, predictions), 3))

### Looking at the neighbours

Because k-NN is just "find the most similar examples", we can ask it **which** books it consulted. Very few models let you do this, and it is worth taking advantage of.

In [ ]:
labels = ["Non-fiction", "Fiction"]
training_titles = list(X_train)

def show_neighbours(title, k=5):
    """ print the k nearest training titles for a given title, and the prediction """
    vector = vectorizer.transform([title])
    distances, indices = knn.kneighbors(vector, n_neighbors=k)

    print(f"QUERY: {title}")
    print(f"   -> predicted: {labels[knn.predict(vector)[0]]}")
    for distance, index in zip(distances[0], indices[0]):
        similarity = 1 - distance
        print(f"   similarity={similarity:.2f}  [{labels[y_train.iloc[index]]:11s}] {training_titles[index][:60]}")


show_neighbours("The Poems of Kent")

### ✏️ Exercise 1

Try `show_neighbours()` on a few titles of your own invention — some that sound like novels, some like local histories. Can you find a title it gets obviously wrong? Look at the neighbours it used: can you see *why* it went wrong?

In [ ]:
# Type your code here:


### Choosing k

`k` is a **hyperparameter**: a setting we choose, rather than something the model learns. Let's try a few values.

In [ ]:
for k in [1, 3, 5, 15, 50, 400]:
    model = KNeighborsClassifier(n_neighbors=k, metric="cosine")
    model.fit(X_train_vectors, y_train)
    score = accuracy_score(y_test, model.predict(X_test_vectors))
    print(f"k={k:3d}  accuracy={score:.3f}")

Neither extreme does well, though the middle of the range is noisier than a textbook diagram — with only 630 training titles, the exact best k moves around a bit run to run. Two things you can reason about without any mathematics, though:

* **k=1** trusts a single nearest book. One eccentric title in the training data and the answer flips — high variance.
* **k=400** — nearly two thirds of the training set — mostly returns whatever label is most common overall. Look at that accuracy: **0.619**, exactly the "always guess non-fiction" baseline we meet properly below.

Small k pays too much attention to individual examples; large k pays too little, until it pays no attention to the title at all. Nearly every knob in machine learning is some version of this trade-off — and real datasets rarely draw as clean a curve as the textbook picture suggests.

## Model 2: Naive Bayes

A different kind of model. k-NN asks *"which books does this one resemble?"* Naive Bayes asks a more direct question: **given the words in this title, which label is more probable — fiction, or non-fiction?**

### From words to probabilities

Genre isn't 50/50 in our corpus — it's already skewed towards non-fiction (about 62%). Knowing nothing else, that skew alone is your best guess. Naive Bayes starts there — this overall split is called the **prior** — and then asks: *how does each word in the title shift the odds?*

Some words are strong evidence: in our training data, a title containing "romance" is fiction far more often than not; a title containing "history" almost never is. Naive Bayes learns exactly how strong that evidence is for every word, then multiplies it all together with the prior to get a probability for each label.

The **"naive"** part: it treats every word as independent evidence, ignoring how words relate to one another in a sentence — a simplification that turns out to work surprisingly well for text.

In one sentence:

> **P(genre | title) is proportional to P(genre) × the product, over every word in the title, of P(word | genre)**


**Example: classifying "The Mysteries of Udolpho"**

Our 630 training titles split about 38% fiction, 62% non-fiction (the same split we meet again as the baseline below). That gives us our starting priors:

- P(Fiction) = 0.38
- P(Non-fiction) = 0.62

Ignoring common words like "the" and "of," the title's content words are *mysteries* and *udolpho*. Suppose counting how often each word appears in fiction vs. non-fiction titles in our training data gives:

| word | P(word \| Fiction) | P(word \| Non-fiction) |
|---|---|---|
| mysteries | 0.02 | 0.001 |
| udolpho | 0.005 | 0.0001 |

Now we plug into the formula for each candidate genre:

- P(Fiction \| title) ∝ 0.38 × 0.02 × 0.005 = 0.000038
- P(Non-fiction \| title) ∝ 0.62 × 0.001 × 0.0001 = 0.000000062

**Fiction scores about 610 times higher**, so the model predicts *Fiction* — which happens to match the actual label.

Notice the word "*proportional to*" doing real work here: these two numbers aren't real probabilities (they don't sum to 1, and don't need to). We only care which one is bigger.


`MultinomialNB` learns those two ingredients straight from the training titles: how common each genre is (the prior), and how often each word occurs in titles of each genre (the **likelihood**).

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
nb.fit(X_train_vectors, y_train)

nb_predictions = nb.predict(X_test_vectors)

print('kNN accuracy:', round(accuracy_score(y_test, predictions), 3))
print("Naive Bayes accuracy:", round(accuracy_score(y_test, nb_predictions), 3))

### Reading the probabilities

Unlike k-NN's plain vote, Naive Bayes gives a probability for each label — `.predict_proba()` returns both, and they always add up to 1. The predicted label is simply whichever one is higher.

In [ ]:
for title in ["The Poems of Kent", "A History of the Parish of Wingham"]:
    p_nonfiction, p_fiction = nb.predict_proba(vectorizer.transform([title]))[0]
    print(f"{title:38s} P(fiction)={p_fiction:.2f}  P(non-fiction)={p_nonfiction:.2f}")

### What did Naive Bayes learn?

For every word, the model stores how likely it is to appear in a fiction title versus a non-fiction one. Comparing those two numbers — which words are disproportionately fiction, which are disproportionately non-fiction — is exactly the kind of pattern a human reader would recognise too.

In [ ]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())

# how much more likely each word is in fiction vs. non-fiction titles
fiction_evidence = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]

top_fiction = np.argsort(fiction_evidence)[-10:][::-1]
top_nonfiction = np.argsort(fiction_evidence)[:10]

print("pushes towards FICTION    :", list(feature_names[top_fiction]))
print("pushes towards NON-FICTION:", list(feature_names[top_nonfiction]))

### Why did it decide *this* title?

We can go one step further and break down a single prediction, word by word. Each word contributes a small amount of evidence towards fiction or non-fiction — `predict_proba()` is just the sum of all that evidence, converted to a probability.

In [ ]:
def explain_nb(title):
    """ show how much evidence each word in a title contributes towards fiction """
    vector = vectorizer.transform([title]).toarray()[0]
    words_used = vector.nonzero()[0]

    print(f"TITLE: {title}")
    for i in words_used:
        evidence = vector[i] * fiction_evidence[i]
        print(f"   {feature_names[i]:15s} {evidence:+.2f}  ({'fiction' if evidence > 0 else 'non-fiction'})")
    print(f"   -> predicted: {labels[nb.predict(vectorizer.transform([title]))[0]]}")
    print()


explain_nb("The Poems of Kent")
explain_nb("A History of the Parish of Wingham")

## Is 0.9 good? The baseline

An accuracy on its own means nothing. You always need something to compare it against.

The simplest comparison: a "model" that ignores the title completely and always answers with whichever label is most common.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train_vectors, y_train)

print("baseline accuracy:", round(accuracy_score(y_test, baseline.predict(X_test_vectors)), 3))

Our corpus is 62% non-fiction, so a model that always says "non-fiction" is right 62% of the time while knowing nothing whatsoever.

**This is why accuracy alone is a dangerous number.** Had the imbalance been sharper — 95% non-fiction, as it would be in many real collections — that useless model would report 95% accuracy.

## Measuring properly: precision, recall, F1

![recpre](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/precision_recall.png)

We need to know how the model does on **each class**, not just overall. Two questions, and they are different:

* **Precision**: when the model says "fiction", how often is it right? *(Do we trust its claims?)*
* **Recall**: of all the fiction that is really there, how much did it find? *(Does it miss things?)*

**F1** combines the two into a single number. The **confusion matrix** shows the raw counts behind them.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, predictions))

print(classification_report(y_test, predictions, target_names=["Non-fiction", "Fiction"]))

Read the confusion matrix as:

```
              predicted non-fiction   predicted fiction
actually non-fiction      correct            wrong
actually fiction           wrong            correct
```

A researcher building a corpus of Victorian novels cares far more about **recall** — the novels the model failed to find are invisible in everything they do afterwards.

## Model 3: a support vector machine

![svm](https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/refs/heads/main/Sessions/images/svm_genre_classification.png)

A **support vector machine** does something different to Naive Bayes — it looks for the **boundary** that best separates fiction from non-fiction, keeping as much clear space as possible on either side.


Imagine plotting every title in our training set as a point, positioned by its features — titles that use similar words end up near each other, fiction titles clustering in one area, non-fiction titles clustering in another.

SVM will try to find the optimal line to separate different classes.

But the SVM doesn't just look for *any* separating line — it looks for the one with the widest possible gap, called the **margin**, between itself and the nearest point of each class.

Here's the part that makes this efficient: to find that widest gap, the algorithm doesn't need to look at every single point. It only needs to look at the points closest to the boundary — the ones "holding up" the margin from either side. These are called the **support vectors**, and they're the only points that actually determine where the line goes. You could delete every other point in the dataset and the boundary wouldn't move at all.

Then a new book is classified by which side of that boundary it falls on.

In [ ]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_vectors, y_train)

svm_predictions = svm.predict(X_test_vectors)

print("k-NN accuracy:", round(accuracy_score(y_test, predictions), 3))
print("Naive Bayes accuracy:", round(accuracy_score(y_test, nb_predictions), 3))
print("SVM accuracy: ", round(accuracy_score(y_test, svm_predictions), 3))
print()
print(classification_report(y_test, svm_predictions, target_names=["Non-fiction", "Fiction"]))

### ✏️ Exercise 2

Above, we vectorised with **TF-IDF**. Go back and build `CountVectorizer()` instead — plain word counts, with no length normalisation and no weighting by rarity — and compare both models.

⚠️ For a fair comparison, keep the other settings the same (`stop_words`, `max_df`, `min_df`, `ngram_range`) — otherwise you're testing more than just TF-IDF vs. plain counts.

Does TF-IDF help both models by about the same amount, or does one benefit more? Can you explain why? (Think about what k-NN actually measures, versus what an SVM can compensate for by learning its own weights.)

In [ ]:
# Type your code here:


## What did the model actually learn?

This is the part that should interest a humanist most. A linear SVM assigns a **weight** to every word: strongly positive words push a title towards fiction, strongly negative towards non-fiction.

We can simply read them off.

In [ ]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
weights = svm.coef_[0]

most_fiction = np.argsort(weights)[-15:][::-1]
most_nonfiction = np.argsort(weights)[:15]

print("pushes towards FICTION    :", list(feature_names[most_fiction]))
print()
print("pushes towards NON-FICTION:", list(feature_names[most_nonfiction]))

No one told the model that *novel*, *tale*, *poems* and *romance* signal fiction, or that *history*, *sketches* and *illustrated* signal non-fiction. It found that in 630 titles.

That list is a research finding in miniature: it is a description, drawn from evidence, of how nineteenth-century books announced their genre on the title page.

### ✏️ Exercise 3: what did it *really* learn?

Now go back and remove the line that filtered to English books, so that all 1,744 labelled books are used. Retrain, and look at the strongest non-fiction weights again.

Among them you will find `og`, `af`, `och`, `van`, `del` — Danish, Swedish and Dutch function words.

What has the model actually learned? And is its accuracy on that version a fair measure of how well it recognises *genre*? This is not a bug in the code. It is a property of the collection, of a kind that is very easy to publish by accident.

In [ ]:
# Type your code here:


## Where this breaks — and where we go next

Our model is good. Now let's break it on purpose, because how a method fails tells you what it is really doing.

Below are pairs of titles. Each pair means nearly the same thing; only one word differs.

In [ ]:
def fiction_score(title):
    """ how confidently does the SVM call this fiction? above 0 = fiction """
    return svm.decision_function(vectorizer.transform([title]))[0]


pairs = [
    ("A Tale of Cornwall",  "A Saga of Cornwall"),
    ("The Poems of Kent",   "The Sonnets of Kent"),
    ("A Novel of Kent",     "A Novella of Kent"),
]

for original, swapped in pairs:
    a, b = fiction_score(original), fiction_score(swapped)
    print(f"{original:22s} {a:+.2f} {labels[int(a > 0)]:12s}   {swapped:22s} {b:+.2f} {labels[int(b > 0)]}")

Swapping *tale* for *saga* — words a reader would treat as near-identical — moves a title from confidently fiction to the wrong side of the boundary.

Why? Look at the weights the model has for those words:

In [ ]:
for word in ["tale", "saga", "poems", "sonnets", "novel", "novella"]:
    if word in vectorizer.vocabulary_:
        print(f"{word:9s} weight = {weights[vectorizer.vocabulary_[word]]:+.2f}")
    else:
        print(f"{word:9s} NOT IN THE VOCABULARY — no column, so it counts for nothing")

There it is. `saga` and `novella` never appeared in the 630 training titles, so the model has **no column for them**. It did not judge them and decide they were weak evidence; it could not see them at all.

And this is not a rare edge case:

In [ ]:
import re

def words_in(texts):
    return set(word for text in texts for word in re.findall(r"[a-z]+", text.lower()))

train_vocabulary = words_in(X_train)
test_vocabulary = words_in(X_test)
unseen = test_vocabulary - train_vocabulary

print(f"{len(unseen)} of the {len(test_vocabulary)} words in the test titles never appeared in training")
print(f"that is {len(unseen) / len(test_vocabulary):.0%} of the test vocabulary, invisible to the model")

**Here is the limitation:** in a bag of words, every word is its own separate column, with no relationship to any other. The model has no way of knowing that a *novella* is a kind of *novel*, that *sonnets* are *poems*, or that a *saga* is a *tale*. Each is just column number 1,472 or column number 89.

To do better, we would need a representation in which **similar words are similar numbers**.

That representation exists, and you already have one in this repository — vectors trained on 1860s newspapers:

(To keep this quick, we are loading a small **extract** of those vectors — a few hundred words, enough for the comparisons below. Tomorrow we load the full model, all 48,054 words of it.)

In [ ]:
!pip install -U gensim

In [ ]:
!wget -q https://raw.githubusercontent.com/Living-with-machines/dhoxss-text2tech/main/Sessions/data/1860s-vectors-sample.txt

from gensim.models import KeyedVectors

vectors = KeyedVectors.load_word2vec_format("1860s-vectors-sample.txt", binary=False)

print("similar meanings:")
for a, b in [("poems", "sonnets"), ("tale", "romance"), ("novel", "romance")]:
    print(f"   similarity({a}, {b}) = {vectors.similarity(a, b):.2f}")

print("\nunrelated words, for comparison:")
for a, b in [("poems", "railway"), ("novel", "boiler"), ("tale", "steam")]:
    print(f"   similarity({a}, {b}) = {vectors.similarity(a, b):.2f}")

print("\nnearest neighbours of 'sonnets':", [w for w, _ in vectors.most_similar("sonnets", topn=6)])

The exact words that broke our classifier — `poems` and `sonnets`, `tale` and `romance` — are **neighbours** in this space, while `poems` and `railway` are far apart. Nobody wrote those relationships down. They were learned from how the words are used.

**Where do such vectors come from, and what else can they do?** 

### One honest warning about that last cell

Try `vectors.most_similar("novella")` and you will get nonsense — Italian names and unrelated words. The 1860s newspapers barely ever used the word, so its vector was learned from almost no evidence.

Word embeddings do not know language. They know the corpus they were trained on. A rare word gets a bad vector, and a corpus with a particular view of the world produces vectors that share it — something to keep firmly in mind tomorrow.

# Solutions

### ✏️ Exercise 1

In [ ]:
show_neighbours("A History of the Parish of Wingham")
print()
show_neighbours("The Mill on the Floss")
print()
show_neighbours("The Sonnets of Kent")

# The classic failure: not every title has 5 genuinely similar neighbours. "Sonnets" only
# shares words with 2 fiction titles in the training set; k-NN still has to fill the other
# 3 "nearest neighbour" slots with something, even titles that share no words at all
# (similarity 0.00). Those 3 filler votes outnumber the 2 relevant ones, and the title
# is misclassified as non-fiction — a side-effect of forcing exactly k neighbours.

### ✏️ Exercise 2

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# same filtering as the TF-IDF vectorizer above, so the only thing that changes is the weighting
count_vectorizer = CountVectorizer(stop_words="english", max_df=0.5, min_df=2,
                                    token_pattern=r"(?u)\b\w\w+\b", ngram_range=(1, 2))
train_counts = count_vectorizer.fit_transform(X_train)
test_counts = count_vectorizer.transform(X_test)

knn_counts = KNeighborsClassifier(n_neighbors=5, metric="cosine").fit(train_counts, y_train)
svm_counts = LinearSVC().fit(train_counts, y_train)

print("with plain counts:")
print("   k-NN:", round(accuracy_score(y_test, knn_counts.predict(test_counts)), 3))
print("   SVM :", round(accuracy_score(y_test, svm_counts.predict(test_counts)), 3))
print("with TF-IDF:")
print("   k-NN:", round(accuracy_score(y_test, predictions), 3))
print("   SVM :", round(accuracy_score(y_test, svm_predictions), 3))

# With the vocabulary filtering held fixed, TF-IDF helps both models by a similar small
# amount here (about +0.02 accuracy each). The classic argument still holds in principle:
# k-NN measures raw similarity, so without IDF a common word counts as much as a rare one,
# while an SVM can learn to downweight common words itself. But our stop-word and max_df
# filtering already remove most of that effect before TF-IDF even gets a turn — when two
# preprocessing choices do overlapping work, isolating which one actually mattered takes
# exactly this kind of controlled, like-for-like comparison.

### ✏️ Exercise 3

In [ ]:
all_books = df[df["genre"].isin(["Fiction", "Non-fiction"])]      # no language filter

Xa = all_books["title"]
ya = (all_books["genre"] == "Fiction").astype(int)
Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    Xa, ya, test_size=0.25, random_state=42, stratify=ya
)

vec_all = TfidfVectorizer()
svm_all = LinearSVC().fit(vec_all.fit_transform(Xa_train), ya_train)

print("accuracy on all languages:", round(accuracy_score(ya_test, svm_all.predict(vec_all.transform(Xa_test))), 3))

names_all = np.array(vec_all.get_feature_names_out())
w_all = svm_all.coef_[0]
print("\npushes towards NON-FICTION:", list(names_all[np.argsort(w_all)[:15]]))

# `og`, `af`, `och`, `van`, `del`, `nyere` are Danish, Swedish and Dutch function words. In
# this collection the Scandinavian and Dutch books are almost all non-fiction, so "is this
# book written in one of those languages?" is a very good predictor of non-fiction — and
# the model happily learned that instead of learning about genre. The accuracy looks fine
# (0.899 — even higher than the English-only model). The model is not doing what we think
# it is doing. Always read the features.